In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import pyspark
import pyspark.sql
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark import SparkConf

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# Sliding windows
def make_sliding_windows(y, history, horizon):
    X, Y = [], []
    for t in range(history, len(y) - horizon):
        X.append(y[t-history:t])
        Y.append(y[t:t+horizon])
    return np.array(X), np.array(Y)

In [ ]:
def split_train_test(X, Y, test_ratio=0.2):
    split = int(len(X) * (1 - test_ratio))
    return X[:split], Y[:split], X[split:], Y[split:]


In [ ]:
def evaluate_forecasts(results):
    yt, yp, lo, up = [], [], [], []

    for r in results:
        yt.extend(r["y_true"])
        yp.extend(r["y_pred"])
        lo.extend(r["lower"])
        up.extend(r["upper"])

    yt, yp, lo, up = map(np.array, [yt, yp, lo, up])

    rmse = np.sqrt(np.mean((yt - yp)**2))
    coverage = np.mean((yt >= lo) & (yt <= up))

    return rmse, coverage


In [ ]:
class DeepAR(nn.Module):
    def __init__(self, hidden_size=64, horizon=28):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            batch_first=True,
            dropout=0.2
        )

        self.dropout = nn.Dropout(0.2)

        self.fc_mu = nn.Linear(hidden_size, horizon)
        self.fc_sigma = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = self.dropout(h[-1])

        mu = self.fc_mu(h)
        sigma = F.softplus(self.fc_sigma(h)) + 1e-6

        return mu, sigma


In [ ]:
def gaussian_nll(mu, sigma, y):
    return torch.mean(
        0.5 * torch.log(2 * torch.pi * sigma**2)
        + 0.5 * ((y - mu)**2) / (sigma**2)
    )

In [ ]:
def plot_forecast(results, i=0):
    r = results[i]

    full = np.concatenate([r["history"], r["y_true"]])
    t_all = np.arange(len(full))
    t_pred = np.arange(len(r["history"]), len(full))

    plt.figure(figsize=(10,4))

    plt.plot(t_all, full, label="Actual", color="black")
    plt.plot(t_pred, r["y_pred"], label="Forecast", color="red")

    plt.fill_between(t_pred, r["lower"], r["upper"], alpha=0.3)

    plt.axvline(len(r["history"]), linestyle="--")

    plt.legend()
    plt.title("Forecast Example")
    plt.show()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

print("✅ Spark OK")

In [ ]:
def run_deepar_pipeline(y, dates):

    # ==========================================
    # Settings
    # ==========================================

    history = 100
    horizon = 28
    epochs = 50

    # ==========================================
    # Log transform
    # ==========================================

    y_log = np.log1p(y)

    X, Y = make_sliding_windows(
        y_log,
        history,
        horizon
    )

    if len(X) == 0:
        print("Series too short.")
        return None

    # ==========================================
    # Train/Test Split
    # ==========================================

    split = int(len(X) * 0.8)

    X_tr = X[:split]
    X_te = X[split:]

    Y_tr = Y[:split]
    Y_te = Y[split:]

    print("Train windows:", len(X_tr))
    print("Test windows :", len(X_te))

    # ==========================================
    # Scaling
    # SAME AS NN
    # ==========================================

    mean = Y_tr.mean()

    std = Y_tr.std() + 1e-8

    X_tr_scaled = (
        X_tr - mean
    ) / std

    X_te_scaled = (
        X_te - mean
    ) / std

    Y_tr_scaled = (
        Y_tr - mean
    ) / std

    Y_te_scaled = (
        Y_te - mean
    ) / std

    # ==========================================
    # Tensors
    # ==========================================

    Xt = torch.tensor(
        X_tr_scaled[..., None],
        dtype=torch.float32
    ).to(device)

    Yt = torch.tensor(
        Y_tr_scaled,
        dtype=torch.float32
    ).to(device)

    Xs = torch.tensor(
        X_te_scaled[..., None],
        dtype=torch.float32
    ).to(device)

    # ==========================================
    # Model
    # ==========================================

    model = DeepAR(
        hidden_size=256,
        horizon=horizon
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    # ==========================================
    # Training
    # ==========================================

    model.train()

    for epoch in range(epochs):

        mu, sigma = model(Xt)

        loss = gaussian_nll(
            mu,
            sigma,
            Yt
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        if epoch % 10 == 0:

            print(
                f"Epoch {epoch:3d} | "
                f"Loss={loss.item():.6f}"
            )

    # ==========================================
    # Prediction
    # ==========================================

    model.eval()

    with torch.no_grad():

        mu, sigma = model(Xs)

    mu = mu.cpu().numpy()
    sigma = sigma.cpu().numpy()

    # ==========================================
    # Prediction Intervals
    # ==========================================

    z = 1.645

    lower = mu - z * sigma
    upper = mu + z * sigma

    # ==========================================
    # Back-transform
    # ==========================================

    mu = mu * std + mean

    lower = lower * std + mean

    upper = upper * std + mean

    Y_te_log = Y_te

    # ==========================================
    # Metrics
    # ==========================================

    rmse = np.sqrt(
        np.mean(
            (Y_te_log - mu) ** 2
        )
    )

    coverage = np.mean(
        (Y_te_log >= lower)
        &
        (Y_te_log <= upper)
    )

    print("\n===== DEEPAR RESULTS =====")

    print(
        "RMSE:",
        round(rmse, 4)
    )

    print(
        "Coverage:",
        round(coverage, 4)
    )

    # ==========================================
    # Store Results + Dates
    # ==========================================

    results = []

    for i in range(len(X_te)):

        history_log = (
            X_te_scaled[i] * std + mean
        )

        origin = split + i

        history_dates = dates[
            origin : origin + history
        ]

        future_dates = dates[
            origin + history :
            origin + history + horizon
        ]

        results.append({

            "history": history_log,

            "y_true": Y_te_log[i],

            "y_pred": mu[i],

            "lower": lower[i],

            "upper": upper[i],

            "history_dates": history_dates,

            "future_dates": future_dates

        })

    # ==========================================
    # Return
    # ==========================================

    return {

        "results_test": results,

        "rmse": rmse,

        "coverage": coverage,

        "history_days": history,

        "forecast_days": horizon,

        "model": model,

        "mean": mean,

        "std": std

    }

In [ ]:

out = run_deepar_pipeline(
    series,
    subset.index
)

In [ ]:
# ==========================================
# CONFORMAL CALIBRATION FOR DEEPAR
# ==========================================

history = out["history_days"]
horizon = out["forecast_days"]

y_log = np.log1p(series)

X_all, Y_all = make_sliding_windows(
    y_log,
    history,
    horizon
)

n = len(X_all)

train_end = int(n * 0.60)
cal_end = int(n * 0.80)

X_cal = X_all[train_end:cal_end]
Y_cal = Y_all[train_end:cal_end]

X_test = X_all[cal_end:]
Y_test = Y_all[cal_end:]

print("Calibration windows:", len(X_cal))
print("Test windows:", len(X_test))

In [ ]:
mean = out["mean"]
std = out["std"]

X_cal_scaled = (
    X_cal - mean
) / std

X_test_scaled = (
    X_test - mean
) / std

In [ ]:
model = out["model"]

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

X_cal_t = torch.tensor(
    X_cal_scaled[..., None],
    dtype=torch.float32
).to(device)

with torch.no_grad():

    mu_cal, sigma_cal = model(
        X_cal_t
    )

mu_cal = mu_cal.cpu().numpy()
sigma_cal = sigma_cal.cpu().numpy()

In [ ]:
z = 1.645

lower_cal = mu_cal - z * sigma_cal
upper_cal = mu_cal + z * sigma_cal

lower_cal = lower_cal * std + mean
upper_cal = upper_cal * std + mean

In [ ]:
cal_scores = np.maximum(
    lower_cal - Y_cal,
    Y_cal - upper_cal
)

In [ ]:
alpha = 0.10

qhat_per_horizon = []

for h in range(horizon):

    qhat = np.quantile(
        cal_scores[:, h],
        1 - alpha,
        method="higher"
    )

    qhat_per_horizon.append(qhat)

qhat_per_horizon = np.array(
    qhat_per_horizon
)

print(qhat_per_horizon)

In [ ]:
X_test_t = torch.tensor(
    X_test_scaled[..., None],
    dtype=torch.float32
).to(device)

with torch.no_grad():

    mu_test, sigma_test = model(
        X_test_t
    )

mu_test = mu_test.cpu().numpy()
sigma_test = sigma_test.cpu().numpy()

In [ ]:
lower_raw = (
    mu_test - z * sigma_test
)

upper_raw = (
    mu_test + z * sigma_test
)

mu_test = mu_test * std + mean

lower_raw = lower_raw * std + mean
upper_raw = upper_raw * std + mean

In [ ]:
lower_conf = (
    lower_raw
    - qhat_per_horizon
)

upper_conf = (
    upper_raw
    + qhat_per_horizon
)

In [ ]:
coverage_raw = np.mean(
    (Y_test >= lower_raw)
    &
    (Y_test <= upper_raw)
)

coverage_conf = np.mean(
    (Y_test >= lower_conf)
    &
    (Y_test <= upper_conf)
)

winkler_raw = compute_winkler_arrays(
    Y_test,
    lower_raw,
    upper_raw
)

winkler_conf = compute_winkler_arrays(
    Y_test,
    lower_conf,
    upper_conf
)

print("\n===== DEEPAR CONFORMAL =====")

print("Raw Coverage :", coverage_raw)
print("Conf Coverage:", coverage_conf)

print("Raw Winkler  :", winkler_raw)
print("Conf Winkler :", winkler_conf)

In [ ]:
# ===================================
# CONFORMAL CALIBRATION
# ===================================

alpha = 0.10

cal_scores = np.maximum(
    lower_cal - Y_cal,
    Y_cal - upper_cal
)

horizon = Y_cal.shape[1]

qhat_per_horizon = []

for h in range(horizon):

    scores_h = cal_scores[:, h]

    qhat_h = np.quantile(
        scores_h,
        1 - alpha,
        method="higher"
    )

    qhat_per_horizon.append(qhat_h)

qhat_per_horizon = np.array(
    qhat_per_horizon
)

print("qhat:")
print(qhat_per_horizon)

In [ ]:
def plot_deepar_conformal_window(
    subset,
    X_test,
    Y_test,
    lower_conf,
    upper_conf,
    origins_test,
    idx,
    title
):

    history = X_test[idx]

    actual = Y_test[idx]

    lower = lower_conf[idx]
    upper = upper_conf[idx]

    pred = (lower + upper) / 2

    origin = int(origins_test[idx])

    history_dates = subset.index[
        origin-len(history):origin
    ]

    future_dates = subset.index[
        origin:origin+len(actual)
    ]

    rmse = np.sqrt(
        np.mean((actual - pred) ** 2)
    )

    coverage = np.mean(
        (actual >= lower)
        &
        (actual <= upper)
    )

    winkler = compute_winkler_arrays(
        actual,
        lower,
        upper
    )

    plt.figure(figsize=(14,6))

    plt.plot(
        history_dates,
        history,
        color="black",
        linewidth=2,
        label="History"
    )

    plt.plot(
        future_dates,
        actual,
        color="blue",
        linewidth=2,
        label="Actual"
    )

    plt.plot(
        future_dates,
        pred,
        color="red",
        linewidth=2,
        label="Forecast"
    )

    plt.fill_between(
        future_dates,
        lower,
        upper,
        color="green",
        alpha=0.25,
        label="Conformal PI"
    )

    plt.axvline(
        history_dates[-1],
        color="red",
        linestyle="--",
        linewidth=2
    )

    plt.title(
        f"{title}\n"
        f"RMSE={rmse:.3f} | "
        f"Coverage={coverage:.3f} | "
        f"Winkler={winkler:.3f}"
    )

    plt.xlabel("Date")
    plt.ylabel("Balance")

    plt.legend()

    plt.xticks(rotation=45)

    plt.tight_layout()

    plt.show()


In [ ]:
history = out["history_days"]
horizon = out["forecast_days"]

y_log = np.log1p(series)

X_all, Y_all = make_sliding_windows(
    y_log,
    history,
    horizon
)

n = len(X_all)

train_end = int(n * 0.60)
cal_end = int(n * 0.80)

origins_all = np.arange(history, history + len(X_all))

origins_test = origins_all[cal_end:]

In [ ]:
deep_ar_conf_metrics = []

for i in range(len(Y_test)):

    actual = Y_test[i]

    pred = (lower_conf[i] + upper_conf[i]) / 2

    lower = lower_conf[i]
    upper = upper_conf[i]

    rmse = np.sqrt(
        np.mean((actual - pred) ** 2)
    )

    coverage = np.mean(
        (actual >= lower)
        &
        (actual <= upper)
    )

    winkler = compute_winkler_arrays(
        actual,
        lower,
        upper
    )

    deep_ar_conf_metrics.append({
        "idx": i,
        "rmse": rmse,
        "coverage": coverage,
        "winkler": winkler
    })

best_conf_idx = min(
    deep_ar_conf_metrics,
    key=lambda x: x["rmse"]
)["idx"]

worst_conf_idx = max(
    deep_ar_conf_metrics,
    key=lambda x: x["rmse"]
)["idx"]

print("Best conformal window :", best_conf_idx)
print("Worst conformal window:", worst_conf_idx)

In [ ]:
plot_deepar_conformal_window(
    subset=subset,
    X_test=X_test,
    Y_test=Y_test,
    lower_conf=lower_conf,
    upper_conf=upper_conf,
    origins_test=origins_test,
    idx=best_conf_idx,
    title="DeepAR Best Conformal Forecast Window"
)

In [ ]:
plot_deepar_conformal_window(
    subset=subset,
    X_test=X_test,
    Y_test=Y_test,
    lower_conf=lower_conf,
    upper_conf=upper_conf,
    origins_test=origins_test,
    idx=worst_conf_idx,
    title="DeepAR Worst Conformal Forecast Window"
)